# Förberedelser

## Importera

In [41]:
import SPARQLWrapper as sparql
import pandas as pd

In [ ]:
endpoint = "https://libris.kb.se/sparql"
#endpoint = "https://libris-qa.kb.se/sparql"

# Analys

## Digitiseringar
Den digitiserade versionen av en fysisk resurs. 

I dagsläget finns en hänvisning, som lokal entitet, till det fysiska originalet i otherPhysicalFormat > describedBy > controlNumber .

Målet är att byta ut den lokala entiteten mot en länakd entitet i reproductionOf.

In [43]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT  ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .
    
  ?instance a :DigitalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))

  FILTER isBLANK(?localEntity) .

  FILTER(regex(?marcDisplayText, "(originalversion|orginalversion)", "i"))

}

'''

digitizations = sparql.get_sparql_dataframe(endpoint, query)
digitizations.info()
digitizations.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 46844 entries, 0 to 46843
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             46844 non-null  str  
 1   recordControlNumber  46844 non-null  str  
 2   localControlNumber   46844 non-null  str  
 3   marcDisplayText      46844 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,recordControlNumber,localControlNumber,marcDisplayText
0,https://libris-qa.kb.se/xg8kbj981c8md3t#it,18151238,111264,Orginalversion:


In [44]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
# digitizations[digitizations.duplicated(keep=False)].to_csv("digitala_med_osynliga_blanknoder.csv")

# Preppa/städa
#digitizations.drop_duplicates(inplace=True)

#digitizations.info()
#digitizations.head(1)

### Finns det digitiseringar som hänviar till flera original? 🤯

In [45]:
repros_with_multiple_originals = digitizations[digitizations.duplicated(subset=["instance"], keep=False)]
repros_with_multiple_originals.value_counts("instance")

instance
https://libris-qa.kb.se/h1t2wj4t55ctmvp#it    2
https://libris-qa.kb.se/dxq0ng3q5066kk4#it    2
https://libris-qa.kb.se/5phq41ch47w3cxl#it    2
https://libris-qa.kb.se/h1t2gj0t4jknmgg#it    2
https://libris-qa.kb.se/0jbqj6hb27c7kx3#it    2
https://libris-qa.kb.se/6qjz3wtj5j7lnc3#it    2
https://libris-qa.kb.se/4ngqfbng28v44c8#it    2
https://libris-qa.kb.se/7rkvkt0k3z4bq61#it    2
https://libris-qa.kb.se/3mfqfq3f5df4ckb#it    2
https://libris-qa.kb.se/4ngrgr3g3s0c545#it    2
https://libris-qa.kb.se/cwp0p09p1q9wnt2#it    2
https://libris-qa.kb.se/zh9l9ks917j3lx7#it    2
https://libris-qa.kb.se/wf7j7jv749kqpdj#it    2
https://libris-qa.kb.se/6qjtjh7j5t3xhpr#it    2
https://libris-qa.kb.se/3mfqfqwf51542d8#it    2
https://libris-qa.kb.se/fzr2r29r5l61rqb#it    2
https://libris-qa.kb.se/l4x7x7gx297jgxb#it    2
https://libris-qa.kb.se/cwp3442p543kz85#it    2
https://libris-qa.kb.se/3mfq5c8f0486gwm#it    2
https://libris-qa.kb.se/n60908f055mgqt8#it    2
https://libris-qa.kb.se/j2v5lsn

Droppa dessa, för att istället hantera manuellt.

In [46]:
digitizations.drop_duplicates(subset="instance", keep=False, inplace=True)
digitizations.info()

<class 'pandas.DataFrame'>
Index: 46796 entries, 0 to 46843
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             46796 non-null  str  
 1   recordControlNumber  46796 non-null  str  
 2   localControlNumber   46796 non-null  str  
 3   marcDisplayText      46796 non-null  str  
dtypes: str(4)
memory usage: 1.8 MB


In [47]:
#repros_with_multiple_originals.to_csv("reproduktioner_med_flera_original.tsv", sep="\t")

## Original
Den fysiska resurs som digitiserats.

In [48]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT ?instance ?recordControlNumber ?localControlNumber ?marcDisplayText WHERE {

	?record :mainEntity ?instance ;
    :controlNumber ?recordControlNumber .

  ?instance a :PhysicalResource ;
    :otherPhysicalFormat ?localEntity .
    # :sameAs ?recordControlNumber .

  FILTER isBLANK(?localEntity) .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localControlNumber, "^[0-9a-z]+$"))
  FILTER(regex(?marcDisplayText, "(^digit.*version)", "i"))
  FILTER(!regex(?note, "(del|1969)", "i"))

}

'''

originals = sparql.get_sparql_dataframe(endpoint, query)
originals.info()
originals.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 45110 entries, 0 to 45109
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             45110 non-null  str  
 1   recordControlNumber  45110 non-null  str  
 2   localControlNumber   45110 non-null  str  
 3   marcDisplayText      45110 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,recordControlNumber,localControlNumber,marcDisplayText
0,https://libris-qa.kb.se/h0sgpknt3vw1r26#it,2612575,p85p20v1m98bhb7n,Digitaliserad version:


In [49]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
#originals[originals.duplicated(keep=False)].to_csv("fysiska_med_osynliga_blanknoder.csv")

# Preppa/städa
#originals.drop_duplicates(inplace=True)

#originals.info()
#originals.head(1)

### Finns det original som pekar mot flera digitiseringar?



In [50]:
originals_with_multiple_repros = originals[originals.duplicated(subset=["instance"], keep=False)]
originals_with_multiple_repros.info()
originals_with_multiple_repros.value_counts("instance")

<class 'pandas.DataFrame'>
Index: 621 entries, 54 to 44970
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   instance             621 non-null    str  
 1   recordControlNumber  621 non-null    str  
 2   localControlNumber   621 non-null    str  
 3   marcDisplayText      621 non-null    str  
dtypes: str(4)
memory usage: 24.3 KB


instance
https://libris-qa.kb.se/xf742w281ml66nx#it    5
https://libris-qa.kb.se/3ld3tvrf3dzzh56#it    4
https://libris-qa.kb.se/9sl8ljsm4d7rfjx#it    4
https://libris-qa.kb.se/cvn7c43p40fm4b7#it    3
https://libris-qa.kb.se/zg8530895flm4dt#it    3
                                             ..
https://libris-qa.kb.se/1jbxgl9c5pw3q32#it    2
https://libris-qa.kb.se/m4xlq8gz5jnzmn6#it    2
https://libris-qa.kb.se/q71p87q22g65xqp#it    2
https://libris-qa.kb.se/7qj6msqk4f14g99#it    2
https://libris-qa.kb.se/cvn989xp0g97pgq#it    2
Name: count, Length: 305, dtype: int64

## Jämförelse

### Hita och skriv ut de ömsesidigt matchande

In [51]:
mutual_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber", "recordControlNumber"],
    right_on=["recordControlNumber", "localControlNumber"],
    suffixes=("_digital", "_physical")
)

# Helt identiska rader kan droppas
mutual_matches.drop_duplicates(inplace=True)

mutual_matches.info()
mutual_matches.head(5)

mutual_matches_to_print = mutual_matches.copy()

LIBRIS_ID_REGEX = r"https://libris(?:-(?:qa|stg))?\.kb\.se/([a-zA-Z0-9]+)#it"

mutual_matches_to_print["instance_digital"] = mutual_matches_to_print["instance_digital"].str.extract(LIBRIS_ID_REGEX, expand=False)
mutual_matches_to_print["instance_physical"] = mutual_matches_to_print["instance_physical"].str.extract(LIBRIS_ID_REGEX, expand=False)
mutual_matches_to_print.to_csv("mutual_matches_ids.tsv", sep="\t", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 43710 entries, 0 to 43709
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              43710 non-null  str  
 1   recordControlNumber_digital   43710 non-null  str  
 2   localControlNumber_digital    43710 non-null  str  
 3   marcDisplayText_digital       43710 non-null  str  
 4   instance_physical             43710 non-null  str  
 5   recordControlNumber_physical  43710 non-null  str  
 6   localControlNumber_physical   43710 non-null  str  
 7   marcDisplayText_physical      43710 non-null  str  
dtypes: str(8)
memory usage: 2.7 MB


### Djupdykning

In [33]:
# Vilka av de ömsesidigt matchande originalen har flera reproduktioner?
mutual_matches[mutual_matches.duplicated(subset=["instance_physical"], keep=False)].sort_values("instance_physical")

,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical
17798,https://libris.kb.se/m5zb7jvz100sc3z#it,19942139,8073130,Orginalversion:,https://libris.kb.se/0h96zbgb5vdzqxh#it,8073130,19942139,Digitaliserad version:
4688,https://libris.kb.se/q779drh3nqx529m7#it,q779drh3nqx529m7,8073130,Originalversion:,https://libris.kb.se/0h96zbgb5vdzqxh#it,8073130,q779drh3nqx529m7,Digitaliserad version:
38231,https://libris.kb.se/vd69hjr61wwk7s9#it,13482516,10254730,Originalversion,https://libris.kb.se/0h99m4gb42hvxzm#it,10254730,13482516,Digitaliserad version
12423,https://libris.kb.se/l4x5w5lx3kmhxfp#it,17259928,10254730,Orginalversion,https://libris.kb.se/0h99m4gb42hvxzm#it,10254730,17259928,Digitaliserad version
11524,https://libris.kb.se/n60c8kx00nlt4xj#it,19942140,305440,Orginalversion:,https://libris.kb.se/0h9wcmpb3ws6ffm#it,305440,19942140,Digitaliserad version:
...,...,...,...,...,...,...,...,...
18603,https://libris.kb.se/3mfwxjbf1h2l55d#it,22548523,1596939,Originalversion:,https://libris.kb.se/zg8wxfn94jp0wjd#it,1596939,22548523,Digitaliserad version:
2142,https://libris.kb.se/vd6736v61wcjnfm#it,12339606,1598259,Originalversion:,https://libris.kb.se/zg8wxg492ws998j#it,1598259,12339606,Digitaliserad version:
2741,https://libris.kb.se/1kcnh4zc3qghn92#it,18219251,1598259,Originalversion,https://libris.kb.se/zg8wxg492ws998j#it,1598259,18219251,Digitaliserad version
40578,https://libris.kb.se/n60cghd013fqnz8#it,20101860,1601799,Originalversion:,https://libris.kb.se/zg8wxl290sqvgts#it,1601799,20101860,Digitaliserad version:


In [34]:
# Vilka reproduktioner har flera original?
mutual_matches[mutual_matches.duplicated(subset=["instance_digital"], keep=False)]

,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical


#### Ensidiga matcher

In [35]:
digi_to_physical_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localControlNumber"],
    right_on=["recordControlNumber"],
    suffixes=("_digital", "_physical")
)

digi_to_physical_matches.drop_duplicates(inplace=True)

digi_to_physical_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 44920 entries, 0 to 44919
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              44920 non-null  str  
 1   recordControlNumber_digital   44920 non-null  str  
 2   localControlNumber_digital    44920 non-null  str  
 3   marcDisplayText_digital       44920 non-null  str  
 4   instance_physical             44920 non-null  str  
 5   recordControlNumber_physical  44920 non-null  str  
 6   localControlNumber_physical   44920 non-null  str  
 7   marcDisplayText_physical      44920 non-null  str  
dtypes: str(8)
memory usage: 2.7 MB


In [36]:
physical_to_digi_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["recordControlNumber"],
    right_on=["localControlNumber"],
    suffixes=("_digital", "_physical")
)

physical_to_digi_matches.drop_duplicates(inplace=True)

physical_to_digi_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 43609 entries, 0 to 43608
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              43609 non-null  str  
 1   recordControlNumber_digital   43609 non-null  str  
 2   localControlNumber_digital    43609 non-null  str  
 3   marcDisplayText_digital       43609 non-null  str  
 4   instance_physical             43609 non-null  str  
 5   recordControlNumber_physical  43609 non-null  str  
 6   localControlNumber_physical   43609 non-null  str  
 7   marcDisplayText_physical      43609 non-null  str  
dtypes: str(8)
memory usage: 2.7 MB


In [37]:
cols = physical_to_digi_matches.columns

non_mutual_matches = pd.concat(
    [physical_to_digi_matches.assign(source="physical_to_digi"),
     digi_to_physical_matches.assign(source="digi_to_physical")]
    ).drop_duplicates(subset=cols, keep=False)

non_mutual_matches.info()
non_mutual_matches.head(1)

<class 'pandas.DataFrame'>
Index: 1631 entries, 128 to 44907
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   instance_digital              1631 non-null   str  
 1   recordControlNumber_digital   1631 non-null   str  
 2   localControlNumber_digital    1631 non-null   str  
 3   marcDisplayText_digital       1631 non-null   str  
 4   instance_physical             1631 non-null   str  
 5   recordControlNumber_physical  1631 non-null   str  
 6   localControlNumber_physical   1631 non-null   str  
 7   marcDisplayText_physical      1631 non-null   str  
 8   source                        1631 non-null   str  
dtypes: str(9)
memory usage: 127.4 KB


,instance_digital,recordControlNumber_digital,localControlNumber_digital,marcDisplayText_digital,instance_physical,recordControlNumber_physical,localControlNumber_physical,marcDisplayText_physical,source
128,https://libris.kb.se/n6016540398z30h#it,11776620,2150409,Originalversion,https://libris.kb.se/l3wj4bfx176p8kv#it,1253278,11776620,Digitaliserad version:,physical_to_digi


- Rader med source "physical_to_digi" representerar fall där en fysisk instans pekar på en digital instans, som inte pekar tillbaka.

- Rader med source "digi_to_physical" representerar fall där en digital instans pekar på en fysisk instans, som inte pekar tillbaka.

In [38]:
non_mutual_matches.value_counts("source")

source
digi_to_physical    1471
physical_to_digi     160
Name: count, dtype: int64

In [39]:
non_mutual_matches.value_counts(["instance_digital", "source"])

instance_digital                          source          
https://libris.kb.se/j2vbjjtv3r846hn#it   physical_to_digi    23
https://libris.kb.se/p71cs88154gv1hj#it   physical_to_digi     4
https://libris.kb.se/dxq1sz9q2j2hxb2#it   digi_to_physical     4
https://libris.kb.se/cwp0rx8p5nnq792#it   digi_to_physical     4
https://libris.kb.se/h1t4w2dt5hx3lv9#it   digi_to_physical     4
                                                              ..
https://libris.kb.se/p1j1kz82m0bn30c2#it  digi_to_physical     1
https://libris.kb.se/gvkb3fjqdtx5rdg2#it  digi_to_physical     1
https://libris.kb.se/r5cbfmc8ph8ghbbx#it  digi_to_physical     1
https://libris.kb.se/7rkvkrrk4w42p7j#it   digi_to_physical     1
https://libris.kb.se/1kcrqt6c3cm7m0j#it   digi_to_physical     1
Name: count, Length: 1553, dtype: int64

In [40]:
non_mutual_matches.to_csv("non_mutual_matches.tsv", sep="\t")